# ASE'22 Tseitin Evaluation

This notebook adapts the original `ase22_evaluation.R` analysis to torte stage outputs. It summarizes transformation/runtime behavior and model-count preservation relative to the Z3 Tseitin baseline.

In [ ]:
from decimal import Decimal, InvalidOperation, getcontext
from pathlib import Path, PurePosixPath
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

evaluation_main = Path(os.getenv("DOCKER_SRC_DIRECTORY", "../../src")) / "evaluation" / "main.py"
exec(open(evaluation_main).read(), globals())

getcontext().prec = 80
BASELINE = "Z3"

TRANSFORM_CSV = Path(get_stage_directory("transform_to_dimacs", key="transform")) / "output.csv"
SAT_CSV = Path(get_stage_directory("solve_sat", key="sat")) / "output.csv"
SHARP_CSV = Path(get_stage_directory("solve_sharp_sat", key="sharp")) / "output.csv"
print("transform:", TRANSFORM_CSV)
print("sat:", SAT_CSV)
print("sharp:", SHARP_CSV)


In [ ]:
TRANSFORM_LABELS = {
    "transform-to-dimacs-with-featureide": "FeatureIDE",
    "transform-to-dimacs-with-kconfigreader": "KConfigReader",
    "transform-smt-to-dimacs-with-z3": "Z3",
}


def to_number(series):
    return pd.to_numeric(series, errors="coerce")


def normalize_transformer(value):
    text = str(value)
    return TRANSFORM_LABELS.get(text, text)


def dimacs_parts(dimacs_file):
    parts = PurePosixPath(str(dimacs_file)).parts
    transformer = parts[0] if parts and parts[0].startswith("transform") else ""
    rest = "/".join(parts[1:] if transformer else parts)
    return pd.Series({
        "dimacs_transformer_from_file": normalize_transformer(transformer),
        "model_key": re.sub(r"\.dimacs$", "", rest),
    })


def solver_short(value):
    text = str(value)
    text = text.split("/")[-1]
    text = text.removesuffix(".sh")
    return text


def query_to_analysis(query, is_sharp):
    query = str(query)
    head = query.split()[0] if query else ""
    if is_sharp:
        return "FMC" if head == "void" else "FC"
    if head in {"core", "dead"}:
        return "Core/Dead"
    return "Void"


def parse_count(value):
    text = str(value).strip()
    if not re.fullmatch(r"[0-9]+", text):
        return pd.NA
    return text


def decimal_ratio(value, baseline):
    try:
        if pd.isna(value) or pd.isna(baseline):
            return np.nan
        den = Decimal(str(baseline))
        if den == 0:
            return np.nan
        return float(Decimal(str(value)) / den)
    except (InvalidOperation, ValueError):
        return np.nan


In [ ]:
transform = pd.read_csv(TRANSFORM_CSV, dtype=str)
transform = transform.join(transform["dimacs_file"].apply(dimacs_parts))
transform["transformation"] = transform["dimacs_transformer"].apply(normalize_transformer)
transform["transform_time"] = to_number(transform.get("dimacs_time"))
transform["transform_variables"] = to_number(transform.get("dimacs_variables"))
transform["transform_literals"] = to_number(transform.get("dimacs_literals"))

transform_summary = (
    transform.groupby(["model_key", "transformation"], dropna=False)
    .agg(
        transform_time=("transform_time", "median"),
        transform_variables=("transform_variables", "median"),
        transform_literals=("transform_literals", "median"),
    )
    .reset_index()
)

baseline_transform = transform_summary[transform_summary["transformation"].eq(BASELINE)][["model_key", "transform_time"]]
baseline_transform = baseline_transform.rename(columns={"transform_time": "baseline_transform_time"})
transform_summary = transform_summary.merge(baseline_transform, on="model_key", how="left")
transform_summary["time_rel"] = transform_summary["transform_time"] / transform_summary["baseline_transform_time"]
transform_summary["analysis"] = "Transformation"

display(transform_summary.head())
print(f"Loaded {len(transform):,} transformation rows for {transform_summary['model_key'].nunique():,} model keys.")


In [ ]:
sat = pd.read_csv(SAT_CSV, dtype=str)
sharp = pd.read_csv(SHARP_CSV, dtype=str)

sat = sat.join(sat["dimacs_file"].apply(dimacs_parts))
sharp = sharp.join(sharp["dimacs_file"].apply(dimacs_parts))

sat["transformation"] = sat["dimacs_transformer_from_file"]
sharp["transformation"] = sharp["dimacs_transformer_from_file"]
sat["solver"] = sat["dimacs_solver"].apply(solver_short)
sharp["solver"] = sharp["dimacs_solver"].apply(solver_short)
sat["analysis"] = sat["dimacs_query"].apply(lambda q: query_to_analysis(q, False))
sharp["analysis"] = sharp["dimacs_query"].apply(lambda q: query_to_analysis(q, True))
sat["solve_time"] = to_number(sat["dimacs_solver_time"])
sharp["solve_time"] = to_number(sharp["dimacs_solver_time"])
sharp["model_count"] = sharp["sharp_sat"].apply(parse_count)

solve = pd.concat([
    sat[["model_key", "transformation", "solver", "analysis", "dimacs_query", "solve_time"]],
    sharp[["model_key", "transformation", "solver", "analysis", "dimacs_query", "solve_time", "model_count"]],
], ignore_index=True)

solve_summary = (
    solve.groupby(["model_key", "transformation", "solver", "analysis"], dropna=False)
    .agg(solve_time=("solve_time", "median"))
    .reset_index()
)
baseline_solve = solve_summary[solve_summary["transformation"].eq(BASELINE)][["model_key", "solver", "analysis", "solve_time"]]
baseline_solve = baseline_solve.rename(columns={"solve_time": "baseline_solve_time"})
solve_summary = solve_summary.merge(baseline_solve, on=["model_key", "solver", "analysis"], how="left")
solve_summary["time_rel"] = solve_summary["solve_time"] / solve_summary["baseline_solve_time"]

display(solve_summary.head())
print(f"Loaded {len(sat):,} SAT rows and {len(sharp):,} #SAT rows.")


In [ ]:
time_data = pd.concat([
    transform_summary[["transformation", "analysis", "time_rel"]],
    solve_summary[["transformation", "analysis", "time_rel"]],
], ignore_index=True)
time_data = time_data.replace([np.inf, -np.inf], np.nan).dropna(subset=["time_rel"])

order = ["Transformation", "Void", "Core/Dead", "FMC", "FC"]
transformations = ["FeatureIDE", "KConfigReader", "Z3"]
fig, ax = plt.subplots(figsize=(12, 5))
positions = []
labels = []
data = []
width = 0.23
for i, analysis in enumerate(order):
    for j, transformation in enumerate(transformations):
        values = time_data[(time_data["analysis"].eq(analysis)) & (time_data["transformation"].eq(transformation))]["time_rel"]
        values = values[(values > 0) & (values < 1e2)]
        if len(values):
            data.append(values.to_numpy())
            positions.append(i + (j - 1) * width)
            labels.append(transformation)
bp = ax.boxplot(data, positions=positions, widths=width * 0.85, patch_artist=True, showfliers=True)
colors = {"FeatureIDE": "#E69F00", "KConfigReader": "#56B4E9", "Z3": "#009E73"}
for patch, label in zip(bp["boxes"], labels):
    patch.set_facecolor(colors[label])
    patch.set_alpha(0.75)
ax.axhline(1, color="black", linestyle="--", linewidth=0.8)
ax.set_yscale("log")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order)
ax.set_xlabel("Algorithm")
ax.set_ylabel("Runtime relative to Z3 baseline (log10)")
handles = [plt.Line2D([0], [0], color=colors[t], lw=8, alpha=0.75) for t in transformations]
ax.legend(handles, transformations, title="CNF Transformation")
ax.set_title("RQ1/RQ2: Relative Runtime")
fig.tight_layout()
fig.savefig("ase22-runtime-relative.pdf")
plt.show()

display(time_data.groupby(["analysis", "transformation"])["time_rel"].describe())


In [ ]:
count_rows = sharp.dropna(subset=["model_count"]).copy()
count_summary = (
    count_rows.groupby(["model_key", "transformation", "solver", "analysis", "dimacs_query"], dropna=False)
    .agg(model_count=("model_count", lambda s: s.dropna().iloc[len(s.dropna()) // 2] if len(s.dropna()) else pd.NA))
    .reset_index()
)
baseline_counts = count_summary[count_summary["transformation"].eq(BASELINE)][[
    "model_key", "solver", "analysis", "dimacs_query", "model_count"
]].rename(columns={"model_count": "baseline_model_count"})
count_summary = count_summary.merge(baseline_counts, on=["model_key", "solver", "analysis", "dimacs_query"], how="left")
count_summary["model_count_rel"] = [
    decimal_ratio(value, baseline)
    for value, baseline in zip(count_summary["model_count"], count_summary["baseline_model_count"])
]
plot_counts = count_summary.replace([np.inf, -np.inf], np.nan).dropna(subset=["model_count_rel"])
plot_counts = plot_counts[(plot_counts["model_count_rel"] > 0) & (plot_counts["model_count_rel"] < 1e2)]

fig, ax = plt.subplots(figsize=(7, 4))
data = []
positions = []
labels = []
for i, transformation in enumerate(transformations):
    for j, analysis in enumerate(["FMC", "FC"]):
        values = plot_counts[(plot_counts["transformation"].eq(transformation)) & (plot_counts["analysis"].eq(analysis))]["model_count_rel"]
        if len(values):
            data.append(values.to_numpy())
            positions.append(i + (j - 0.5) * 0.25)
            labels.append(analysis)
bp = ax.boxplot(data, positions=positions, widths=0.2, patch_artist=True, showfliers=True)
analysis_colors = {"FMC": "#0072B2", "FC": "#D55E00"}
for patch, label in zip(bp["boxes"], labels):
    patch.set_facecolor(analysis_colors[label])
    patch.set_alpha(0.75)
ax.axhline(1, color="black", linestyle="--", linewidth=0.8)
ax.set_yscale("log")
ax.set_xticks(range(len(transformations)))
ax.set_xticklabels(transformations)
ax.set_xlabel("CNF Transformation")
ax.set_ylabel("Model count relative to Z3 baseline (log10)")
handles = [plt.Line2D([0], [0], color=analysis_colors[a], lw=8, alpha=0.75) for a in ["FMC", "FC"]]
ax.legend(handles, ["FMC", "FC"], title="Analysis")
ax.set_title("RQ3: Model Count Preservation")
fig.tight_layout()
fig.savefig("ase22-model-count-relative.pdf")
plt.show()

display(count_summary.groupby(["analysis", "transformation"])["model_count_rel"].describe())
